# KaggleLink · Kaggle host

Connect Claude Code on your computer to Ollama on this Kaggle session.

**Before starting:** enable internet and select a suitable GPU. Install Ollama 0.14.0+ and cloudflared using their official installation instructions. Keep Ollama bound to loopback. Pull a model with tool-calling support. This notebook does not install system binaries or download a model automatically.

Upload or clone the KaggleLink source to `/kaggle/working/kagglelink`. Change the path below if needed. Never enter credentials in notebook source. Configure Kaggle Secrets named `KAGGLELINK_TOKEN` and, for discovery, `KAGGLELINK_GITHUB_TOKEN`, then enable notebook access to those secrets.


In [ ]:
%pip install -q /kaggle/working/kagglelink


In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["KAGGLELINK_TOKEN"] = secrets.get_secret("KAGGLELINK_TOKEN")

# Optional automatic discovery. Use a dedicated PRIVATE repo, not the public source repo.
REGISTRY_REPO = ""  # Example: "your-user/kagglelink-sessions"
if REGISTRY_REPO:
    os.environ["KAGGLELINK_GITHUB_TOKEN"] = secrets.get_secret("KAGGLELINK_GITHUB_TOKEN")
print("Secrets loaded without displaying their values.")


## Check prerequisites

Start `ollama serve` separately if it is not already running. If you need to pull a model, use `ollama pull MODEL` before continuing. Match the context window and model size to your GPU memory. On a typical Kaggle notebook, use loopback Ollama at port 11434.


In [ ]:
import shutil
import subprocess
import urllib.request
import json

assert shutil.which("ollama"), "Install Ollama first"
assert shutil.which("cloudflared"), "Install cloudflared first"
with urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=5) as response:
    print("Ollama:", json.load(response)["version"])
subprocess.run(["ollama", "list"], check=True)


In [ ]:
import sys

MODEL = "YOUR_INSTALLED_MODEL"  # Use an exact name from ollama list.
assert MODEL != "YOUR_INSTALLED_MODEL", "Set MODEL to an installed model first"
command = [sys.executable, "-m", "kagglelink", "host", "--model", MODEL]
if REGISTRY_REPO:
    command += ["--repo", REGISTRY_REPO]

# This cell stays running to keep the bridge and tunnel alive.
# Interrupt it when finished. If the tunnel is left running after interruption,
# stop the notebook session before starting another host on the same port.
subprocess.run(command, check=True)


## On your own computer

Install the same source, set the same `KAGGLELINK_TOKEN`, and launch Claude Code:

```powershell
python -m kagglelink launch --url https://THE-URL-PRINTED-ABOVE.trycloudflare.com
```

With automatic discovery, set the PC's read-only `KAGGLELINK_GITHUB_TOKEN`, then use:

```powershell
python -m kagglelink launch --repo your-user/kagglelink-sessions
```

Run `/status` inside Claude and confirm `http://127.0.0.1:17654`. Keep this Kaggle session running. Kaggle quotas and inactivity limits still apply. Never publish notebook outputs containing credentials.
